In [ ]:
# Built for a different version of the simulations, so not expected to run without modification.
# Some visualizations here may be worth reviving so now fully deleting yet.
# from myst_nb import glue
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
import simlib.generate_sims
import simlib.generate_timeseries
from IPython.display import HTML

from tedana.decay import fit_decay_ts

In [ ]:
# User-provided parameter initializations
te_baseline=28
s0_baseline=5
r2s_baseline=1/14
s0_r2s_prop_list = np.linspace(0, 1, 21)
middle_prop_idx = np.round(21/2).astype(int)
minlist_prop_idx = np.array([0, 5, 10, 15, 20])
print(f"{s0_r2s_prop_list=}")
te_list = np.linspace(0, 60, 16)
te16_idx = np.where(te_list==16)[0][0].astype(int)
te28_idx = np.where(te_list==28)[0][0].astype(int)
te44_idx = np.where(te_list==44)[0][0].astype(int)
te16_28_44_idx = np.array([te16_idx, te28_idx, te44_idx]).astype(int)
print(f"{te_list=}")
noise_list = [0]# , 0.04, 0.08, 0.12, 0.16]

In [ ]:
def pchange(values, means):
    return 100*(values-means)/means

In [ ]:
# Demonstrations of basic properties
pchange_range=50
s_input_linear = np.linspace(-pchange_range, pchange_range, 201) # a line of percent change values
s_zero_idx = np.where(s_input_linear==0)[0][0].astype(int)
s_n1_idx = 0
s_p1_idx = len(s_input_linear)-1
s_input_3_idx = np.array([s_n1_idx, s_zero_idx, s_p1_idx])
print(s_zero_idx)
demo_timepoints = len(s_input_linear)
demo_data = np.zeros((demo_timepoints, len(te_list), len(s0_r2s_prop_list)))

for s0_prop_idx, proportion_s0 in enumerate(s0_r2s_prop_list):
    for echo_idx, te_output in enumerate(te_list):
        demo_data[:,echo_idx, s0_prop_idx] = simlib.generate_sims.generate_s_with_noise(
            te_baseline=te_baseline,
            s0_baseline=s0_baseline,
            r2s_baseline=r2s_baseline,
            inputted_s=s_input_linear,
            proportion_s0_r2s=proportion_s0,
            te_output=te_output,
            noise_var_ratio=0,
            pchange=False
            )
        

# calculate t2s and s0 values from simulated data
demo_adaptive_mask = ((len(te_list))*np.ones(len(s_input_linear))).astype(int)

t2s, s0, failures, t2s_var, s0_var, t2s_s0_covar = fit_decay_ts(demo_data, te_list, demo_adaptive_mask, fittype="curvefit", n_threads=1)
print(t2s.shape, s0.shape)

# baseline_demo_data = np.squeeze(demo_data[s_zero_idx,:,:])
# p1_demo_data = np.squeeze(demo_data[s_p1_idx,:,:])
# n1_demo_data = np.squeeze(demo_data[s_n1_idx,:,:])    

plt.figure(figsize=(10,14))
plt.subplot(4,2,1)
plt.plot(s_input_linear, np.squeeze(demo_data[:,te28_idx,minlist_prop_idx]))
plt.ylabel(f"Raw Signal, TE=28")
plt.xlabel("% change of input signal")
plt.subplot(4,2,2)
plt.plot(s_input_linear, pchange(np.squeeze(demo_data[:,te28_idx,minlist_prop_idx]), np.squeeze(demo_data[s_zero_idx,te28_idx,minlist_prop_idx])))
plt.ylabel(f"%change from input=0, TE=28")
plt.xlabel("% change of input signal")
plt.subplot(4,2,3)
plt.plot(s_input_linear, np.squeeze(demo_data[:,te16_idx,minlist_prop_idx]))
plt.ylabel(f"Raw Signal, TE=14")
plt.xlabel("% change of input signal")
plt.subplot(4,2,4)
plt.plot(s_input_linear, pchange(np.squeeze(demo_data[:,te16_idx,minlist_prop_idx]), np.squeeze(demo_data[s_zero_idx,te16_idx,minlist_prop_idx])))
plt.ylabel(f"%change from input=0, TE=14")
plt.xlabel("% change of input signal")
plt.subplot(4,2,5)
plt.plot(s_input_linear, np.squeeze(demo_data[:,te44_idx,minlist_prop_idx]))
plt.ylabel(f"Raw Signal, TE=44")
plt.xlabel("% change of input signal")
plt.subplot(4,2,6)
plt.plot(s_input_linear, pchange(np.squeeze(demo_data[:,te44_idx,minlist_prop_idx]), np.squeeze(demo_data[s_zero_idx,te44_idx,minlist_prop_idx])))
plt.ylabel(f"%change from input=0, TE=44")
plt.legend(s0_r2s_prop_list[minlist_prop_idx], fontsize='xx-small', title="1=only dS0, 0=only dR2*" )
plt.xlabel("% change of input signal")
plt.show()

def regression_line(x, y, color='black'):
    print(f"{x.shape=}, {y.shape=}")
    linear_y = np.full(y.shape, np.nan)
    for idx in range(y.shape[1]):
        coeffs = np.polyfit(x, y[:,idx], 1)
        linear_y[:,idx] = coeffs[0]*x + coeffs[1]
    plt.plot(x, linear_y, linestyle='--')

plt.figure(figsize=(10,14))
plt.subplot(4,2,1)
regression_line(s0_r2s_prop_list, np.squeeze(demo_data[s_input_3_idx,te28_idx,:].T))
plt.plot(s0_r2s_prop_list, np.squeeze(demo_data[s_input_3_idx,te28_idx,:].T))
plt.ylabel(f"Raw Signal, TE=28")
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.subplot(4,2,2)
regression_line(s0_r2s_prop_list, pchange(np.squeeze(demo_data[s_input_3_idx,te28_idx,:]), np.squeeze(demo_data[s_zero_idx,te28_idx,:])).T)
plt.plot(s0_r2s_prop_list, pchange(np.squeeze(demo_data[s_input_3_idx,te28_idx,:]), np.squeeze(demo_data[s_zero_idx,te28_idx,:])).T)
plt.ylabel(f"%change from input=0, TE=28")
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.subplot(4,2,3)
regression_line(s0_r2s_prop_list, np.squeeze(demo_data[s_input_3_idx,te16_idx,:].T))
plt.plot(s0_r2s_prop_list, np.squeeze(demo_data[s_input_3_idx,te16_idx,:].T))
plt.ylabel(f"Raw Signal, TE=16")
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.subplot(4,2,4)
regression_line(s0_r2s_prop_list, pchange(np.squeeze(demo_data[s_input_3_idx,te16_idx,:]), np.squeeze(demo_data[s_zero_idx,te16_idx,:])).T)
plt.plot(s0_r2s_prop_list, pchange(np.squeeze(demo_data[s_input_3_idx,te16_idx,:]), np.squeeze(demo_data[s_zero_idx,te16_idx,:])).T)
plt.ylabel(f"%change from input=0, TE=16")
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.subplot(4,2,5)
regression_line(s0_r2s_prop_list, np.squeeze(demo_data[s_input_3_idx,te44_idx,:].T))
plt.plot(s0_r2s_prop_list, np.squeeze(demo_data[s_input_3_idx,te44_idx,:].T))
plt.ylabel(f"Raw Signal, TE=44")
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.subplot(4,2,6)
regression_line(s0_r2s_prop_list, pchange(np.squeeze(demo_data[s_input_3_idx,te44_idx,:]), np.squeeze(demo_data[s_zero_idx,te44_idx,:])).T)
plt.plot(s0_r2s_prop_list, pchange(np.squeeze(demo_data[s_input_3_idx,te44_idx,:]), np.squeeze(demo_data[s_zero_idx,te44_idx,:])).T)
plt.ylabel(f"%change from input=0, TE=44")
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.legend(s_input_linear[s_input_3_idx], fontsize='xx-small', title="% change from mean" )
plt.show()


plt.figure(figsize=(10,10))
plt.subplot(2,2,1)
plt.plot(te_list, np.squeeze(demo_data[s_n1_idx,:,minlist_prop_idx].T))
plt.ylabel(f"Raw Signal, -1% inputted S change")
plt.xlabel("echo times")
plt.subplot(2,2,2)
plt.plot(te_list, pchange(np.squeeze(demo_data[s_n1_idx,:,minlist_prop_idx]), np.squeeze(demo_data[s_zero_idx,:,minlist_prop_idx])).T)
plt.ylabel(f"%pchange from change=0 to -1%")
plt.xlabel("echo times")
plt.subplot(2,2,3)
plt.plot(te_list, np.squeeze(demo_data[s_p1_idx,:,minlist_prop_idx].T))
plt.ylabel(f"Raw Signal, 1% inputted S change")
plt.xlabel("echo times")
plt.subplot(2,2,4)
plt.plot(te_list, pchange(np.squeeze(demo_data[s_p1_idx,:,minlist_prop_idx]), np.squeeze(demo_data[s_zero_idx,:,minlist_prop_idx])).T)
plt.ylabel(f"%pchange from change=0 to 1%")
plt.xlabel("echo times")
plt.legend(s0_r2s_prop_list[minlist_prop_idx],fontsize='xx-small', title="1=only dS0, 0=only dR2*" )
plt.show()



plt.figure(figsize=(6,20))
plt.subplot(2,4,1)
plt.plot(s_input_linear, 1/t2s[:,minlist_prop_idx])
plt.xlabel("% change of input signal")
plt.ylabel("R2* estimate")

plt.subplot(2,4,3)
plt.plot(s_input_linear, t2s[:,minlist_prop_idx])
plt.xlabel("% change of input signal")
plt.ylabel("T2* estimate")
plt.subplot(2,4,4)
plt.plot(s_input_linear, s0[:,minlist_prop_idx])
plt.xlabel("% change of input signal")
plt.ylabel("S0* estimate")
plt.subplot(2,4,5)
plt.plot(s0_r2s_prop_list, 1/t2s[s_input_3_idx,:].T)
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.ylabel("R2* estimate")
plt.subplot(2,4,7)
plt.plot(s0_r2s_prop_list, t2s[s_input_3_idx,:].T)
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.ylabel("T2* estimate")
plt.subplot(2,4,8)
plt.plot(s0_r2s_prop_list, s0[s_input_3_idx,:].T)
plt.xlabel("Proportion of deltaS0/deltaR2s")
plt.ylabel("S0* estimate")
plt.show()





In [ ]:
# Creating baseline random timeseries
n_reps=5
n_timepoints=300
s_input = simlib.generate_timeseries.gen_bandpass_randn_timeseries(n_reps=n_reps, n_timepoints=n_timepoints)
plt.plot(s_input[0:2,:].T)

In [ ]:
sim_data = np.zeros((n_reps, n_timepoints, len(te_list), len(s0_r2s_prop_list), len(noise_list)))

for noise_idx, noise_var_ratio in enumerate(noise_list):
    for s0_prop_idx, proportion_s0 in enumerate(s0_r2s_prop_list):
        for echo_idx, te_output in enumerate(te_list):
            sim_data[:,:,echo_idx, s0_prop_idx, noise_idx] = simlib.generate_sims.generate_s_with_noise(
                te_baseline=te_baseline,
                s0_baseline=s0_baseline,
                r2s_baseline=r2s_baseline,
                inputted_s=s_input,
                proportion_s0_r2s=proportion_s0,
                te_output=te_output,
                noise_var_ratio=noise_var_ratio,
                pchange=False
                )

In [ ]:
rep_idx = 0
noise_idx = 0
s0_prop_idx = 0


# gif code based on https://www.geeksforgeeks.org/create-an-animated-gif-using-python-matplotlib/
fig, axes = plt.subplots(
    nrows=2, figsize=(14, 10), gridspec_kw={"height_ratios": [1, 3]}
)

# Set constant params for figure
axes[0].set_ylabel("Magnitude", fontsize=24)
axes[0].set_xlabel("Echoes", fontsize=24)
#axes[0].set_xlim(0, N_VOLS - 1)
#axes[0].tick_params(axis="both", which="major", labelsize=14)
# axes[1].set_ylabel("Signal", fontsize=24)
# axes[1].set_xlabel("Echo Time (ms)", fontsize=24)
# axes[1].set_xticks(echo_times)
# axes[1].set_ylim(0, np.ceil(np.max(fullcurve_signal) / 1000) * 1000)
# axes[1].set_xlim(0, np.max(FULLCURVE_TES))
# axes[1].tick_params(axis="both", which="major", labelsize=14)
fig.tight_layout()

ax0_line_plot = axes[0].plot(te_list, np.squeeze(sim_data[rep_idx,0,:, s0_prop_idx, noise_idx]), color="black", zorder=0)
# ax0_scatter_plot = axes[0].scatter(
#     0,
#     singleecho_signal[:, 0],
#     color="orange",
#     s=150,
#     label="Single-Echo Signal",
#     zorder=1,
# )
# ax1_scatter_plot = axes[1].scatter(
#     SINGLEECHO_TE,
#     singleecho_signal[:, 0],
#     color="orange",
#     s=150,
#     alpha=1.0,
#     label="Single-Echo Signal",
#     zorder=1,
# )


def AnimationFunction(frame):
    """Function takes frame as an input."""
    ax0_line_plot[0].set_data(
        (
            te_list,
            np.squeeze(sim_data[rep_idx,frame,:, s0_prop_idx, noise_idx]),
        )
    )

anim_created = FuncAnimation(fig, AnimationFunction, frames=int(sim_data.shape[1]/3), interval=100)
# anim_created.save()
# plt.show()
html = HTML(anim_created.to_jshtml())
# glue("fig_single-echo", html, display=False)

In [ ]:
rep_idx = 0
noise_idx = 0
s0_prop_idx=0 #len(s0_r2s_prop_list)-1
plt.plot(te_list, np.squeeze(sim_data[rep_idx,0:50,:, s0_prop_idx, noise_idx]).T)

In [ ]:
# calculate t2s and s0 time series from simulated data
noise_idx=0
rep_idx=0
# Making fit_date with samples, echoes, time dimensions
# In this case, samples is the s0_prop_list dimension
fit_data = sim_data[rep_idx,:,:, :, noise_idx].transpose(2,1,0)
print(f"{fit_data.shape=}")
adaptive_mask = ((len(te_list))*np.ones(len(s0_r2s_prop_list))).astype(int)

t2s, s0, failures, t2s_var, s0_var, t2s_s0_covar = fit_decay_ts(fit_data, te_list, adaptive_mask, fittype="curvefit", n_threads=8)

In [ ]:
#t2s, s0, failures, t2s_var, s0_var, t2s_s0_covar
print(t2s.shape)

plt.subplot(3,1,1)
plt.plot(s0[-1,:]/ np.mean(s0[-1,:]))
plt.plot(t2s[0,:] / np.mean(t2s[0,:]))
plt.subplot(3,1,2)
plt.plot(np.std(t2s,axis=1))
plt.subplot(3,1,3)
plt.plot(np.std(s0,axis=1))
plt.show()


fit_data, te_list
te28_idx = np.where(te_list==28)
print(te28_idx)
fit_data28 = np.squeeze(fit_data[:,te28_idx,:])
print(fit_data28.shape)

one_index = np.where((s_input.flatten()>0.99) & (s_input.flatten()<1.01))
print(one_index[0][0])
print(f"{s_input.flatten()[one_index[0][0]]=}\n{t2s.flatten()[one_index[0][0]]=} {s0.flatten()[one_index[0][0]]=}\n{fit_data28.flatten()[one_index[0][0]]=}")
zero_index = np.where((s_input.flatten()>-0.0001) & (s_input.flatten()<0.0001))
print(zero_index[0][0].shape)
print(f"{s_input.flatten()[zero_index[0][0]]=}\n{t2s.flatten()[zero_index[0][0]]=} {s0.flatten()[zero_index[0][0]]=}\n{fit_data28.flatten()[zero_index[0][0]]=}")


print(f"{np.max(fit_data28.flatten())=} {np.min(fit_data28.flatten())=}")
#print(f"{np.max(t2s.flatten())=} {np.max(s0.flatten())=}")

# plt.hist(s_input.flatten(), bins=50)
#plt.show()

In [ ]:
zero_index[0][0]

In [ ]:
rep_idx=1
noise_idx=0
plt.figure(figsize=(20,20))
for s0_prop_idx, proportion_s0 in enumerate(s0_r2s_prop_list):
    plt.subplot(7,3,s0_prop_idx+1)
    plt.plot(np.squeeze(sim_data[rep_idx,:,:,s0_prop_idx,noise_idx])/np.mean(sim_data[rep_idx,:,:,s0_prop_idx,noise_idx], axis=0, keepdims=True))
    plt.title(f"S0/R2* proportion {proportion_s0}")

In [ ]:
rep_idx=1
noise_idx=0

plt.figure(figsize=(20,20))
for s0_prop_idx, proportion_s0 in enumerate(s0_r2s_prop_list):
    plt.subplot(7,3,s0_prop_idx+1)
    plt.plot(np.log(np.squeeze(sim_data[rep_idx,:,:,s0_prop_idx,noise_idx]).T))
    plt.title(f"S0/R2* proportion {proportion_s0}")

In [ ]:
rep_idx=1
s0_prop_idx=0
plt.figure(figsize=(20,20))
for noise_idx, noise_var_ratio in enumerate(noise_list):
    plt.subplot(2,3,noise_idx+1)
    plt.plot(np.squeeze(sim_data[rep_idx,:,:,s0_prop_idx,noise_idx]))
    plt.title(f"Noise level {noise_var_ratio}")

In [ ]:
sim_data

In [ ]:
p = np.linspace(0,1,100)
TE=28
s0_baseline=500
r2s_baseline=1/28
ln_s=np.log(p)*np.log(s0_baseline) - np.log(1-p)*TE*r2s_baseline
plt.plot(p, (ln_s))

In [ ]:
S = np.linspace(-10,10,50)
n_S = len(S)
delta_s0 = np.linspace(-50,50,100)
n_ds0 = len(delta_s0)
p = np.linspace(0,0.99,20)
n_p = len(p)
TE=28
s0_baseline=500
r2s_baseline=1/28

delta_r2s = np.zeros((n_S, n_ds0, n_p))
for ds_idx, ds_val in enumerate(delta_s0):
    for p_idx, p_val in enumerate(p):
        delta_r2s[:,ds_idx, p_idx] = (p_val * ds_val / s0_baseline - S) / ((1-p_val) * TE) 


plt.plot(delta_s0, delta_r2s[:,:,8].T)


In [ ]:
X =np.random.randn(1000000)+3
Z = 2*np.random.randn(1000000)+5

p=0.3
for p in [0.1, 0.3, 0.5, 0.7, 0.9]:
    Y2 = (p/(1-p))*(X**2)
    print(p, np.mean((X**2)*Y2))


print(np.var(X), np.var(Y), np.var(X*Y))
print(np.mean(X**2) - np.mean(X)**2)
print(np.mean(Y**2) - np.mean(Y)**2)
print(np.mean(X**2), np.mean(Y**2), np.mean(X)**2, np.mean(Y)**2)
print(np.mean(X**2)*np.mean(Y**2) - (np.mean(X)**2)*(np.mean(Y)**2))
print(np.mean((X**2)*(Y**2)) - (np.mean(X*Y)**2))

In [ ]:
s0_r2s_prop_list[0:21:5]